# WayTooCooked — RL Training Notebook

This notebook trains a DQN agent from scratch to make a burger in the Overcooked-like kitchen environment.

**Key improvements over a naive setup:**
- **Reward shaping**: intermediate rewards for each sub-task milestone prevent sparse-reward issues
- **Milestone tracking**: each reward is given only once per episode to prevent reward hacking
- **Bug fixes**: empty-space interaction, chopping-board observation sync, delivery penalty
- **DQN from scratch**: experience replay, target network, ε-greedy exploration

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import random
from collections import deque


## Environment

The `OvercookedKitchen` environment with all bug fixes and reward shaping applied.

In [ ]:
class OvercookedKitchen(gym.Env):
    def __init__(self, layout):
        self.layout = layout
        self.height = len(layout)
        self.width = len(layout[0])
        self.TIME_TO_CHOP = 3
        self.TIME_TO_COOK = 5
        self.MAX_STEPS = 200

        # Actions : 0-3 Movement, 4 Interact, 5 Chop
        self.action_space = spaces.Discrete(6)
        # Observation: agent_pos (2), agent_dir (2), held_obj (1), pan_state (1),
        #              dish_state (3), chop_board (1), step_count (1) = 11 values
        self.observation_space = spaces.Box(
            low=-1, high=max(self.width, self.height, self.MAX_STEPS),
            shape=(11,), dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.step_count = 0
        self.agent_pos = (2, 2)
        self.agent_dir = (0, 1)
        self.held_obj = None
        self.done = False

        self.objects_on_counters = {}
        self.pans_state = {}
        self.dishes = {}
        self.chopping_boards = {}

        # Milestone tracking: each sub-task reward is given only once per episode
        self.milestones = set()

        for y, row in enumerate(self.layout):
            for x, tile in enumerate(row):
                if tile == 'P':
                    self.pans_state[(x, y)] = {'status': 'empty', 'timer': 0}
                elif tile == 'D':
                    self.dishes[(x, y)] = {'bread': False, 'tomato': False, 'steak': False}
                elif tile == 'C':
                    self.chopping_boards[(x, y)] = 0
        return self._get_obs(), {}

    def _milestone_reward(self, milestone_name, reward_value):
        """Return reward_value the first time a milestone is reached, 0 afterwards."""
        if milestone_name not in self.milestones:
            self.milestones.add(milestone_name)
            return reward_value
        return 0.0

    def render(self):
        grid_copy = [list(row) for row in self.layout]
        for (x, y), obj in self.objects_on_counters.items():
            char = obj[0].upper() if isinstance(obj, str) else obj['name'][0].upper()
            grid_copy[y][x] = char
        ax, ay = self.agent_pos
        dir_map = {(0, -1): '^', (0, 1): 'v', (-1, 0): '<', (1, 0): '>'}
        grid_copy[ay][ax] = dir_map.get(self.agent_dir, 'A')
        print("\n" + "="*20)
        for row in grid_copy:
            print(" ".join(row))
        print(f"Hand : {self.held_obj}")
        print(f"Pans : {[(p['status'], p['timer']) for p in self.pans_state.values()]}")
        print(f"Dishes : {[(d['bread'], d['tomato'], d['steak']) for d in self.dishes.values()]}")
        print(f"Chopping boards : {[(c, self.chopping_boards[c]) for c in self.chopping_boards]}")
        print(f"Steps: {self.step_count}/{self.MAX_STEPS}")
        print("="*20)

    def _get_obs(self):
        agent_info = [self.agent_pos[0], self.agent_pos[1], self.agent_dir[0], self.agent_dir[1]]

        obj_map = {None: 0, 'tomato': 1, 'steak': 2, 'bread': 3,
                   'chopped_tomato': 4, 'cooked_steak': 5, 'full_burger': 6}
        if isinstance(self.held_obj, dict):
            held_val = obj_map.get(self.held_obj.get('name'), 0)
        else:
            held_val = obj_map.get(self.held_obj, 0)

        # Pan state: 0=empty, 1..TIME_TO_COOK-1=cooking timer, TIME_TO_COOK=ready
        pan_states = []
        for pos in self.pans_state:
            pan = self.pans_state[pos]
            if pan['status'] == 'ready':
                pan_states.append(self.TIME_TO_COOK)
            else:
                pan_states.append(pan['timer'])

        dish_states = []
        for pos in self.dishes:
            d = self.dishes[pos]
            dish_states.extend([int(d['bread']), int(d['tomato']), int(d['steak'])])

        # Chopping board: reflects actual chopping progress (synced in _handle_chopping)
        chop_states = [self.chopping_boards[pos] for pos in self.chopping_boards]

        obs = np.array(agent_info + [held_val] + pan_states + dish_states + chop_states + [float(self.step_count)],
                       dtype=np.float32)
        return obs

    def _get_adj_pos(self):
        x, y = self.agent_pos
        dx, dy = self.agent_dir
        return (x + dx, y + dy)

    def step(self, action):
        reward = -0.01  # Survival penalty

        # Passive cooking progress
        for _, pan in self.pans_state.items():
            if pan['status'] == 'cooking':
                pan['timer'] += 1
                if pan['timer'] >= self.TIME_TO_COOK:
                    pan['status'] = 'ready'
                    reward += self._milestone_reward('cooked_steak_ready', 1.0)

        if action < 4:  # MOVEMENT
            self._handle_move(action)

        elif action == 4:  # INTERACT
            target_pos = self._get_adj_pos()
            tile = self.layout[target_pos[1]][target_pos[0]]

            if tile == 'T':
                if self.held_obj is None:
                    self.held_obj = 'tomato'
                    reward += self._milestone_reward('picked_tomato', 0.5)
            elif tile == 'S':
                if self.held_obj is None:
                    self.held_obj = 'steak'
                    reward += self._milestone_reward('picked_steak', 0.5)
            elif tile == 'B':
                if self.held_obj is None:
                    self.held_obj = 'bread'
                    reward += self._milestone_reward('picked_bread', 0.5)
            elif tile == 'P':
                reward += self._handle_pan_interact(target_pos)
            elif tile == ' ':  # BUG FIX: was '' which never matched
                self._handle_empty_space_interact(target_pos)
            elif tile == 'D':
                reward += self._handle_dish_interact(target_pos)
            elif tile == 'C':
                reward += self._handle_chopping_board_interact(target_pos)
            elif tile == 'V':
                reward += self._handle_delivery()

        elif action == 5:  # CHOP
            target_pos = self._get_adj_pos()
            if self.layout[target_pos[1]][target_pos[0]] == 'C':
                reward += self._handle_chopping(target_pos)

        self.step_count += 1
        if self.step_count >= self.MAX_STEPS:
            self.done = True

        return self._get_obs(), reward, self.done, False, {}

    def _handle_move(self, action):
        directions = [(0, -1), (0, 1), (-1, 0), (1, 0)]
        dx, dy = directions[action]
        self.agent_dir = (dx, dy)
        new_x = self.agent_pos[0] + dx
        new_y = self.agent_pos[1] + dy
        if 0 <= new_x < self.width and 0 <= new_y < self.height:
            if self.layout[new_y][new_x] == ' ':
                self.agent_pos = (new_x, new_y)

    def _handle_chopping_board_interact(self, target_pos):
        obj_on_board = self.objects_on_counters.get(target_pos)
        reward = 0.0
        if obj_on_board is None and self.held_obj == 'tomato':
            self.objects_on_counters[target_pos] = {'name': 'tomato', 'status': 'raw', 'progression': 0}
            self.held_obj = None
            self.chopping_boards[target_pos] = 0
            reward += self._milestone_reward('placed_tomato_on_board', 0.5)
        elif obj_on_board is not None and obj_on_board['status'] == 'chopped' and self.held_obj is None:
            self.held_obj = 'chopped_tomato'
            del self.objects_on_counters[target_pos]
            self.chopping_boards[target_pos] = 0
            reward += self._milestone_reward('picked_chopped_tomato', 0.5)
        return reward

    def _handle_empty_space_interact(self, target_pos):
        obj_on_tile = self.objects_on_counters.get(target_pos)
        if self.held_obj is not None:
            if obj_on_tile is None:
                self.objects_on_counters[target_pos] = self.held_obj
                self.held_obj = None
        elif obj_on_tile is not None:
            self.held_obj = obj_on_tile
            del self.objects_on_counters[target_pos]

    def _handle_pan_interact(self, target_pos):
        pan = self.pans_state.get(target_pos)
        reward = 0.0
        if self.held_obj == 'steak' and pan['status'] == 'empty':
            pan['status'] = 'cooking'
            pan['timer'] = 0
            self.held_obj = None
            reward += self._milestone_reward('placed_steak_in_pan', 0.5)
        elif pan['status'] == 'ready' and self.held_obj is None:
            self.held_obj = 'cooked_steak'
            pan['status'] = 'empty'
            pan['timer'] = 0
            reward += self._milestone_reward('picked_cooked_steak', 0.5)
        return reward

    def _handle_dish_interact(self, target_pos):
        dish = self.dishes.get(target_pos)
        reward = 0.0
        if self.held_obj:
            if self.held_obj == 'bread' and not dish['bread']:
                dish['bread'] = True
                self.held_obj = None
                reward += self._milestone_reward('placed_bread_on_dish', 2.0)
            elif self.held_obj == 'chopped_tomato' and not dish['tomato'] and dish['bread']:
                dish['tomato'] = True
                self.held_obj = None
                reward += self._milestone_reward('placed_tomato_on_dish', 2.0)
            elif self.held_obj == 'cooked_steak' and not dish['steak'] and dish['bread']:
                dish['steak'] = True
                self.held_obj = None
                reward += self._milestone_reward('placed_steak_on_dish', 2.0)
        elif all(dish.values()):
            self.held_obj = 'full_burger'
            for key in dish:
                dish[key] = False
            reward += self._milestone_reward('assembled_burger', 2.0)
        return reward

    def _handle_chopping(self, target_pos):
        obj = self.objects_on_counters.get(target_pos)
        reward = 0.0
        if obj and obj['name'] == 'tomato' and obj['status'] == 'raw':
            obj['progression'] += 1
            self.chopping_boards[target_pos] = obj['progression']  # BUG FIX: sync observation
            if obj['progression'] >= self.TIME_TO_CHOP:
                obj['status'] = 'chopped'
                reward += self._milestone_reward('chopped_tomato_done', 1.0)
        return reward

    def _handle_delivery(self):
        if self.held_obj == 'full_burger':
            self.held_obj = None
            self.done = True
            return 10.0
        else:
            return -1.0  # Increased penalty to deter wrong deliveries


In [ ]:
# Layout Legend:
# X: Wall/Counter (impassable)
# T: Tomatoes, S: Steaks, B: Bread
# C: Chopping board, P: Pan, D: Dish
#  : Empty space (where one can walk and place objects)

MY_LAYOUT = [
    "X T S B X",
    "C       P",
    "X       X",
    "X   D   V",
    "XXXXXXXXX"
]

# Sanity check
env = OvercookedKitchen(MY_LAYOUT)
obs, _ = env.reset()
print(f"Observation space: {env.observation_space}")
print(f"Action space:      {env.action_space}")
print(f"Initial obs:       {obs}")


## DQN Agent (from scratch with NumPy)

We implement a full Deep Q-Network from scratch using only NumPy:
- **Two-layer neural network** with ReLU activations
- **Experience replay** to break temporal correlations
- **Target network** for training stability
- **ε-greedy** exploration with exponential decay

In [ ]:
class NeuralNetwork:
    """A simple two-layer fully-connected network, implemented in pure NumPy."""

    def __init__(self, input_size, hidden_size, output_size, lr=1e-3):
        # He initialization for ReLU layers
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros(hidden_size)
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros(output_size)
        self.lr = lr
        # Cached values for backprop
        self._x = self._z1 = self._a1 = self._z2 = None

    def forward(self, x):
        """x: (batch, input_size) → Q-values: (batch, output_size)"""
        self._x = x
        self._z1 = x @ self.W1 + self.b1
        self._a1 = np.maximum(0.0, self._z1)  # ReLU
        self._z2 = self._a1 @ self.W2 + self.b2
        return self._z2

    def backward(self, targets):
        """MSE loss backward pass; updates weights in-place."""
        batch_size = self._x.shape[0]
        d2 = 2.0 * (self._z2 - targets) / batch_size
        dW2 = self._a1.T @ d2
        db2 = d2.sum(axis=0)
        d1 = (d2 @ self.W2.T) * (self._z1 > 0)  # ReLU gradient
        dW1 = self._x.T @ d1
        db1 = d1.sum(axis=0)
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2

    def copy_weights_from(self, other):
        """Hard-copy weights from another NeuralNetwork (for target network updates)."""
        self.W1 = other.W1.copy()
        self.b1 = other.b1.copy()
        self.W2 = other.W2.copy()
        self.b2 = other.b2.copy()


In [ ]:
class ReplayBuffer:
    """Fixed-size circular buffer for experience replay."""

    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            np.array(states, dtype=np.float32),
            np.array(actions, dtype=np.int32),
            np.array(rewards, dtype=np.float32),
            np.array(next_states, dtype=np.float32),
            np.array(dones, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buffer)


In [ ]:
class DQNAgent:
    """
    Deep Q-Network agent with:
    - Online Q-network updated every step
    - Target network updated every `target_update_freq` training steps
    - ε-greedy exploration with exponential decay
    - Experience replay buffer
    """

    def __init__(
        self,
        state_size,
        action_size,
        hidden_size=256,
        lr=5e-4,
        gamma=0.99,
        epsilon_start=1.0,
        epsilon_min=0.05,
        epsilon_decay=0.997,
        buffer_size=20000,
        batch_size=128,
        target_update_freq=20,
    ):
        self.action_size = action_size
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        self._train_steps = 0

        self.q_net = NeuralNetwork(state_size, hidden_size, action_size, lr)
        self.target_net = NeuralNetwork(state_size, hidden_size, action_size, lr)
        self.target_net.copy_weights_from(self.q_net)

        self.memory = ReplayBuffer(buffer_size)

    def select_action(self, state):
        """ε-greedy action selection."""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.action_size)
        q_values = self.q_net.forward(state[np.newaxis, :])[0]
        return int(np.argmax(q_values))

    def store(self, state, action, reward, next_state, done):
        self.memory.push(state, action, reward, next_state, done)

    def train(self):
        """One gradient update step; returns MSE loss (or None if buffer too small)."""
        if len(self.memory) < self.batch_size:
            return None

        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)

        # Bellman targets
        next_q = self.target_net.forward(next_states)
        max_next_q = next_q.max(axis=1)
        td_targets = rewards + (1.0 - dones) * self.gamma * max_next_q

        # Update only the Q-value for the taken action
        current_q = self.q_net.forward(states)
        targets = current_q.copy()
        targets[np.arange(self.batch_size), actions] = td_targets

        loss = float(np.mean((current_q - targets) ** 2))
        self.q_net.backward(targets)

        self._train_steps += 1
        if self._train_steps % self.target_update_freq == 0:
            self.target_net.copy_weights_from(self.q_net)

        # Decay exploration
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        return loss

    def save(self, path):
        np.savez(path, W1=self.q_net.W1, b1=self.q_net.b1,
                 W2=self.q_net.W2, b2=self.q_net.b2)

    def load(self, path):
        data = np.load(path)
        self.q_net.W1, self.q_net.b1 = data['W1'], data['b1']
        self.q_net.W2, self.q_net.b2 = data['W2'], data['b2']
        self.target_net.copy_weights_from(self.q_net)


## Training

Train the DQN agent for 2000 episodes and track:
- **Total reward per episode** (shaped + terminal)
- **Success rate** (episode ended with burger delivery)
- **Exploration rate** ε over time

In [ ]:
np.random.seed(42)
random.seed(42)

env = OvercookedKitchen(MY_LAYOUT)
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

agent = DQNAgent(
    state_size=state_size,
    action_size=action_size,
    hidden_size=256,
    lr=5e-4,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_min=0.05,
    epsilon_decay=0.997,
    buffer_size=20000,
    batch_size=128,
    target_update_freq=20,
)

N_EPISODES = 2000
rewards_history = []
success_history = []
epsilon_history = []
LOG_EVERY = 100

for episode in range(N_EPISODES):
    state, _ = env.reset()
    total_reward = 0.0
    success = False

    for _ in range(env.MAX_STEPS):
        action = agent.select_action(state)
        next_state, reward, done, truncated, _ = env.step(action)
        agent.store(state, action, reward, next_state, done or truncated)
        agent.train()
        state = next_state
        total_reward += reward
        if done:
            success = True
            break
        if truncated:
            break

    rewards_history.append(total_reward)
    success_history.append(float(success))
    epsilon_history.append(agent.epsilon)

    if (episode + 1) % LOG_EVERY == 0:
        avg_r = np.mean(rewards_history[-LOG_EVERY:])
        sr = np.mean(success_history[-LOG_EVERY:]) * 100
        print(f"Episode {episode+1:4d} | Avg reward: {avg_r:7.2f} | "
              f"Success rate: {sr:5.1f}% | ε: {agent.epsilon:.3f}")

print("\nTraining complete!")
print(f"Final success rate (last {LOG_EVERY} eps): {np.mean(success_history[-LOG_EVERY:])*100:.1f}%")


## Results

We smooth the raw per-episode curves with a rolling average to visualise learning progress.

In [ ]:
WINDOW = 50
kernel = np.ones(WINDOW) / WINDOW
smoothed_rewards = np.convolve(rewards_history, kernel, mode='valid')
smoothed_success = np.convolve(success_history, kernel, mode='valid') * 100

episodes = np.arange(len(smoothed_rewards)) + WINDOW

# Reward curve
print("=" * 50)
print("Reward per episode (smoothed, last 20 values):")
for i, r in enumerate(smoothed_rewards[-20:]):
    bar = '#' * max(0, int((r + 2) * 2))
    ep = len(rewards_history) - 20 + i + WINDOW
    print(f"  Ep {ep:4d}: {r:7.2f}  {bar}")

print()
print("Success rate (smoothed, last 20 values):")
for i, s in enumerate(smoothed_success[-20:]):
    bar = '#' * int(s / 2)
    ep = len(success_history) - 20 + i + WINDOW
    print(f"  Ep {ep:4d}: {s:5.1f}%  {bar}")

print()
print(f"Best single-episode reward:  {max(rewards_history):.2f} (ep {np.argmax(rewards_history)+1})")
print(f"Final avg reward (last {WINDOW} ep): {np.mean(rewards_history[-WINDOW:]):.2f}")
print(f"Final success rate (last {WINDOW} ep): {np.mean(success_history[-WINDOW:])*100:.1f}%")


## Greedy Evaluation

Run a single episode with ε = 0 (fully greedy) to see what the trained agent does.

In [ ]:
eval_env = OvercookedKitchen(MY_LAYOUT)
state, _ = eval_env.reset()
eval_env.render()

saved_epsilon = agent.epsilon
agent.epsilon = 0.0  # Fully greedy

total_eval_reward = 0.0
for step in range(eval_env.MAX_STEPS):
    action = agent.select_action(state)
    action_names = ['Up', 'Down', 'Left', 'Right', 'Interact', 'Chop']
    next_state, reward, done, truncated, _ = eval_env.step(action)
    eval_env.render()
    print(f"Step {step+1:3d}: {action_names[action]:<10} | reward: {reward:+.3f}")
    total_eval_reward += reward
    state = next_state
    if done or truncated:
        break

agent.epsilon = saved_epsilon  # Restore

print()
outcome = 'SUCCESS 🎉' if 'assembled_burger' in eval_env.milestones else 'did not complete burger'
print(f"Evaluation result: {outcome}")
print(f"Total evaluation reward: {total_eval_reward:.2f}")
print(f"Milestones reached: {eval_env.milestones}")
